# Phase 0b — Pipeline Diagnostics

Investigates 3 blockers from Phase 0:

1. **Missing running-indicator data** — `RV2:BTPU2BPDRUM.AG` and `RV3:TXSU3TS15A.AG`
   have 0 rows in `gold.fact_pi` under their asset_id even though the bridge says
   they should be there.
2. **RV3 NULL Timestamps** — 99.4% of RV3 PI rows have NULL `Timestamp`. RV2 is fine.
3. **BFP missing running indicator** — confirm there's no row in `gold.running_indicator`
   for `RV3_U3_Boiler_Feed_Pump_East`.

All conclusions written to stdout in clearly-marked verdict sections.

## Blocker #1a — Do the running-indicator tags exist in RAW pi_data?

Raw table is `pi_data` (no schema), columns include `Tag`, `Timestamp`, `ValueNumeric`.
We check three places:
1. Raw pi_data (the bronze table)
2. `bridge_pi_tag_to_asset` (the authoritative intake bridge)
3. `gold.fact_pi` (what made it through)

In [ ]:
from pyspark.sql import functions as F

RUN_TAGS = ["RV2:BTPU2BPDRUM.AG", "RV3:TXSU3TS15A.AG"]
print("Checking running-indicator tags:", RUN_TAGS)

print("\n[1] Raw pi_data:")
raw = spark.table("pi_data")
print(f"  pi_data schema columns: {raw.columns}")
(raw.filter(F.col("Tag").isin(RUN_TAGS))
    .groupBy("Tag")
    .agg(F.count("*").alias("rows"),
         F.count("ValueNumeric").alias("numeric_rows"),
         F.count("Timestamp").alias("non_null_ts"),
         F.min("Timestamp").alias("first_ts"),
         F.max("Timestamp").alias("last_ts"))
    .show(truncate=False))

print("\n[2] bridge_pi_tag_to_asset (intake authoritative):")
(spark.table("gold.bridge_pi_tag_to_asset")
    .filter(F.col("Tag").isin(RUN_TAGS))
    .show(truncate=False))

print("\n[3] gold.fact_pi (post-build):")
(spark.table("gold.fact_pi")
    .filter(F.col("Tag").isin(RUN_TAGS))
    .groupBy("Tag","asset_id","pi_match_source")
    .agg(F.count("*").alias("rows"),
         F.count("ValueNumeric").alias("numeric_rows"))
    .show(truncate=False))

## Blocker #1b — Tag-name fuzzy search

Maybe the running-tag names in the intake checklist are slightly off vs raw
`pi_data`. Search raw pi_data for tags that look similar.

In [ ]:
# Pull distinct prefixes that look related
print("RV2 tags containing 'BPDRUM' or 'DRUM' (likely drum-pressure variants):")
(raw.filter(F.col("Tag").rlike("(?i)BPDRUM|DRUM"))
    .filter(F.col("Tag").startswith("RV2:"))
    .groupBy("Tag").count().orderBy(F.col("count").desc()).show(30, truncate=False))

print("RV3 tags containing 'TS15' or 'TS1' + 'A' (turbine shaft speed variants):")
(raw.filter(F.col("Tag").rlike("(?i)TS15|TXSU3"))
    .filter(F.col("Tag").startswith("RV3:"))
    .groupBy("Tag").count().orderBy(F.col("count").desc()).show(30, truncate=False))

## Blocker #3a — Where do the RV3 NULL Timestamps come from?

In Phase 0 we saw RV3 has a single "day = NULL" row representing 14M rows with
NULL Timestamp. Verify in raw pi_data: is the NULL coming from the bronze layer
or being introduced during the gold join?

In [ ]:
# Raw pi_data NULL Timestamp breakdown by plant prefix
prefix_null = (raw
    .withColumn("prefix", F.substring("Tag", 1, 3))
    .groupBy("prefix")
    .agg(F.count("*").alias("rows"),
         F.sum(F.col("Timestamp").isNull().cast("int")).alias("null_ts"),
         F.sum(F.col("Timestamp").isNotNull().cast("int")).alias("good_ts"))
    .withColumn("null_pct", F.round(100.0 * F.col("null_ts") / F.col("rows"), 2)))

print("Raw pi_data NULL Timestamp by Tag prefix:")
prefix_null.orderBy("prefix").show(truncate=False)

In [ ]:
# For RV3, which specific tags have NULL Timestamps vs not
print("RV3 tags with NULL Timestamp counts:")
rv3_null = (raw
    .filter(F.col("Tag").startswith("RV3:"))
    .groupBy("Tag")
    .agg(F.count("*").alias("total"),
         F.sum(F.col("Timestamp").isNull().cast("int")).alias("null_ts"),
         F.sum(F.col("Timestamp").isNotNull().cast("int")).alias("good_ts"))
    .withColumn("null_pct", F.round(100.0 * F.col("null_ts") / F.col("total"), 1))
    .cache())

print(f"Total distinct RV3 tags: {rv3_null.count()}")
print("\nTop 20 RV3 tags by NULL count:")
rv3_null.orderBy(F.col("null_ts").desc()).show(20, truncate=False)

print("\nRV3 tags with 0% NULL (Timestamps look OK):")
print(f"  count: {rv3_null.filter(F.col('null_pct') == 0).count()}")
print("\nRV3 tags with 100% NULL:")
print(f"  count: {rv3_null.filter(F.col('null_pct') == 100).count()}")
print("\nRV3 tags with mixed (some NULL some not):")
print(f"  count: {rv3_null.filter((F.col('null_pct') > 0) & (F.col('null_pct') < 100)).count()}")

## Blocker #3b — What does a sample of NULL-Timestamp RV3 rows look like?

If the row is otherwise complete (Tag + ValueNumeric present, just Timestamp NULL),
ingest is dropping the timestamp parse. If the row is mostly NULL, the source file
itself is malformed.

In [ ]:
print("Sample of RV3 rows with NULL Timestamp:")
rv3_bad = (raw
    .filter(F.col("Tag").startswith("RV3:"))
    .filter(F.col("Timestamp").isNull())
    .limit(20))
rv3_bad.show(truncate=False)

print("\nValue-column non-null pattern in RV3 NULL-Timestamp rows:")
(raw.filter(F.col("Tag").startswith("RV3:"))
    .filter(F.col("Timestamp").isNull())
    .agg(F.count("*").alias("rows"),
         F.count("Tag").alias("with_tag"),
         F.count("ValueNumeric").alias("with_num"),
         F.count("ValueString").alias("with_str"),
         F.count("IsSystem").alias("with_issys"))
    .show(truncate=False))

## Blocker #3c — Source file granularity

`pi_data` was loaded from JSON.GZ files in `Files/PIData/`. If the table has a
file-source column we can isolate which file batches produced the NULLs. If not,
we'll have to look at the bronze loader.

If `pi_data` doesn't expose source filename, check whether the underlying
parquet has `input_file_name()`.

In [ ]:
print("pi_data columns:", raw.columns)

# Try input_file_name() to see which underlying parquet file each row came from
print("\nRV3 NULL-Timestamp rows grouped by source parquet file (sample):")
try:
    by_file = (raw
        .filter(F.col("Tag").startswith("RV3:"))
        .filter(F.col("Timestamp").isNull())
        .withColumn("_src", F.input_file_name())
        .groupBy("_src").count()
        .orderBy(F.col("count").desc())
        .limit(10))
    by_file.show(truncate=False)
except Exception as e:
    print(f"  input_file_name not available: {e}")

## Blocker #2 — BFP running indicator

Confirm there's no row for `RV3_U3_Boiler_Feed_Pump_East` in `gold.running_indicator`.

In [ ]:
print("All gold.running_indicator rows:")
spark.table("gold.running_indicator").show(truncate=False)

print("\nRV3 tags that look like 'pump running' candidates (suction/discharge pressure, motor amps, speed):")
(raw
    .filter(F.col("Tag").startswith("RV3:"))
    .filter(F.col("Tag").rlike("(?i)FP.*PB|FP.*AMP|FP.*RPM|FP.*SPEED|BFP|FEEDPUMP"))
    .groupBy("Tag").count()
    .orderBy(F.col("count").desc())
    .show(30, truncate=False))

## Verdict

Pulls everything into a single answer block.

In [ ]:
print("=" * 72)
print("PHASE 0b VERDICT")
print("=" * 72)

# Re-run the diagnostics inline as verdict booleans
b_rv2_in_raw = raw.filter(F.col("Tag") == "RV2:BTPU2BPDRUM.AG").count() > 0
b_rv3_in_raw = raw.filter(F.col("Tag") == "RV3:TXSU3TS15A.AG").count() > 0

b_rv2_in_bridge = (spark.table("gold.bridge_pi_tag_to_asset")
    .filter(F.col("Tag") == "RV2:BTPU2BPDRUM.AG").count() > 0)
b_rv3_in_bridge = (spark.table("gold.bridge_pi_tag_to_asset")
    .filter(F.col("Tag") == "RV3:TXSU3TS15A.AG").count() > 0)

rv2_null_pct = (raw.filter(F.col("Tag").startswith("RV2:"))
    .agg((100.0 * F.sum(F.col("Timestamp").isNull().cast("int")) / F.count("*")).alias("p"))
    .first()["p"]) or 0.0
rv3_null_pct = (raw.filter(F.col("Tag").startswith("RV3:"))
    .agg((100.0 * F.sum(F.col("Timestamp").isNull().cast("int")) / F.count("*")).alias("p"))
    .first()["p"]) or 0.0

bfp_has_ri = (spark.table("gold.running_indicator")
    .filter(F.col("asset_id") == "RV3_U3_Boiler_Feed_Pump_East").count() > 0)

print(f"\nBlocker #1a -- Running-tag existence in raw pi_data:")
print(f"  RV2:BTPU2BPDRUM.AG in raw pi_data:    {b_rv2_in_raw}")
print(f"  RV3:TXSU3TS15A.AG  in raw pi_data:    {b_rv3_in_raw}")
print(f"  RV2:BTPU2BPDRUM.AG in intake bridge:  {b_rv2_in_bridge}")
print(f"  RV3:TXSU3TS15A.AG  in intake bridge:  {b_rv3_in_bridge}")

print(f"\nBlocker #3 -- NULL Timestamp rate in raw pi_data:")
print(f"  RV2: {rv2_null_pct:.2f}%")
print(f"  RV3: {rv3_null_pct:.2f}%")

print(f"\nBlocker #2 -- BFP running indicator present: {bfp_has_ri}")

print("\nNext-action matrix:")
print("  - Running-tag missing from raw pi_data    -> need a new PI export")
print("  - Running-tag missing from intake bridge  -> edit sensor_tags.csv + rerun intake")
print("  - RV3 NULL Timestamps in raw              -> bronze ingest needs a fix")
print("  - BFP indicator missing                   -> SME-supplied data gap")